# Preprocessing 
This notebook prepares the raw dataset for machine learning:
- Drops identifiers (e.g., `session_id`)
- Encodes categorical features (One-Hot)
- Winsorizes numeric outliers (1st–99th percentile) + Robust scaling
- Creates train/test split (stratified)
- Saves preprocessed CSVs + the fitted preprocessing pipeline

**Tip:** This preprocessing design follows the EDA notes.

## 1) Imports

In [7]:
import numpy as np
import pandas as pd

from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.base import clone
from src.preprocessing.custom_transformers import Winsorizer

## 2) Paths

In [8]:
CURRENT_DIR = Path().resolve()

PROJECT_ROOT = CURRENT_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
FINAL_DATA_DIR = DATA_DIR / "final"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = FINAL_DATA_DIR / "cybersecurity_intrusion_data_v1.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_PATH:", DATASET_PATH)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)


PROJECT_ROOT: C:\Users\Asus\PycharmProjects\AIProject
DATASET_PATH: C:\Users\Asus\PycharmProjects\AIProject\data\final\cybersecurity_intrusion_data_v1.csv
PROCESSED_DATA_DIR: C:\Users\Asus\PycharmProjects\AIProject\data\processed


## 3) Load dataset

In [9]:
df = pd.read_csv(DATASET_PATH, keep_default_na=False)
display(df.head())
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)


,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,599,TCP,4,492.983263,DES,0.606818,1,Edge,0,1
1,SID_00002,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0,0
2,SID_00003,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0,1
3,SID_00004,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0,1
4,SID_00005,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0,0


Shape: (9537, 11)

Dtypes:
session_id              object
network_packet_size      int64
protocol_type           object
login_attempts           int64
session_duration       float64
encryption_used         object
ip_reputation_score    float64
failed_logins            int64
browser_type            object
unusual_time_access      int64
attack_detected          int64
dtype: object


## 4) Quick data quality checks

In [10]:
print("Missing values per column:")
display(df.isna().sum().sort_values(ascending=False))

print("\nLabel distribution (attack_detected):")
display(df["attack_detected"].value_counts(normalize=True).rename("ratio"))


Missing values per column:


session_id             0
network_packet_size    0
protocol_type          0
login_attempts         0
session_duration       0
encryption_used        0
ip_reputation_score    0
failed_logins          0
browser_type           0
unusual_time_access    0
attack_detected        0
dtype: int64


Label distribution (attack_detected):


attack_detected
0    0.552899
1    0.447101
Name: ratio, dtype: float64

**Observation:**  
No true missing values are present in the dataset. All columns report zero null entries, confirming that values such as `"None"` in the `encryption_used` feature are preserved as valid categorical values rather than being treated as missing data.


## 5) Define preprocessing
Decisions:
- Drop `session_id` because it is an identifier.
- Numerical: median imputation → winsorize (clip) outliers → RobustScaler.
- Categorical: most-frequent imputation → OneHotEncoder.

Why winsorize + RobustScaler? Your EDA showed outliers in `network_packet_size` and `session_duration`; this reduces their influence without deleting rows.

In [14]:
TARGET_COL = "attack_detected"
DROP_COLS = ["session_id"]

# 1) Build X, y
X = df.drop(columns=[TARGET_COL] + DROP_COLS)
y = df[TARGET_COL]

# 2) Detect numeric/categorical columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

ENCRYPTION_COL = "encryption_used"
enc_cols = [ENCRYPTION_COL] if ENCRYPTION_COL in cat_cols else []
other_cat_cols = [c for c in cat_cols if c != ENCRYPTION_COL]

print("Numeric columns:", num_cols)
print("Categorical columns (excluding encryption_used):", other_cat_cols)
print("Encryption column:", enc_cols)

# 3) Pipelines
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("winsor", Winsorizer(0.01, 0.99)),
    ("scaler", RobustScaler()),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

encryption_pipe = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# 4) ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, other_cat_cols),
        ("enc", encryption_pipe, enc_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print(" Preprocessor built.")
print("X shape:", X.shape, "| y shape:", y.shape)

Numeric columns: ['network_packet_size', 'login_attempts', 'session_duration', 'ip_reputation_score', 'failed_logins', 'unusual_time_access']
Categorical columns (excluding encryption_used): ['protocol_type', 'browser_type']
Encryption column: ['encryption_used']
 Preprocessor built.
X shape: (9537, 9) | y shape: (9537,)


**Observation:**  
The preprocessing components and feature groups were successfully defined. Numeric features are prepared with median imputation, winsorization to control extreme outliers, and robust scaling. Categorical features are encoded using One-Hot Encoding, with the `encryption_used` column handled separately to preserve `"None"` as a valid category.


## 6) Train/Test split (stratified)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("\nTrain label ratio:")
display(y_train.value_counts(normalize=True))
print("\nTest label ratio:")
display(y_test.value_counts(normalize=True))


Train size: (7629, 9) Test size: (1908, 9)

Train label ratio:


attack_detected
0    0.55289
1    0.44711
Name: proportion, dtype: float64


Test label ratio:


attack_detected
0    0.552935
1    0.447065
Name: proportion, dtype: float64

**Observation:**  
The dataset was successfully split into training and testing sets using stratified sampling. The class proportions of `attack_detected` are preserved across both subsets, confirming that the split did not introduce class distribution bias.


## 7) Fit + transform

In [16]:
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_p, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_p, columns=feature_names, index=X_test.index)

train_processed = pd.concat([X_train_df, y_train], axis=1)
test_processed = pd.concat([X_test_df, y_test], axis=1)

display(train_processed.head())
print("Processed train shape:", train_processed.shape)
print("Processed test shape:", test_processed.shape)


,network_packet_size,login_attempts,session_duration,ip_reputation_score,failed_logins,unusual_time_access,protocol_type_ICMP,protocol_type_TCP,protocol_type_UDP,browser_type_Chrome,browser_type_Edge,browser_type_Firefox,browser_type_Safari,browser_type_Unknown,encryption_used_AES,encryption_used_DES,encryption_used_None,attack_detected
3334,1.048507,1.0,-0.409824,0.707637,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0
4684,-0.768657,0.5,1.452758,-0.168986,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
6385,0.951493,-1.0,0.534196,-0.405144,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0
7288,0.649254,0.0,-0.621728,0.870927,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1
8466,0.455224,0.0,2.500439,-0.581826,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0


Processed train shape: (7629, 18)
Processed test shape: (1908, 18)


**Observation:**  
The preprocessing pipeline was fitted on the training data and applied consistently to the test set, resulting in fully numeric feature matrices with identical feature spaces. Both processed datasets have the expected number of samples and features, confirming that the transformation was applied correctly without data leakage.


In [17]:
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

out_path = PROJECT_ROOT / "data" / "processed" / "preprocessor.joblib"
out_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, out_path)
print(" Saved fitted preprocessor to:", out_path)

 Saved fitted preprocessor to: C:\Users\Asus\PycharmProjects\AIProject\data\processed\preprocessor.joblib


In [18]:
p = joblib.load(out_path)
print("feature count:", len(p.get_feature_names_out()))
print(" preprocessor is fitted and loadable")

feature count: 17
 preprocessor is fitted and loadable


## 8) Save artifacts
Outputs:
- `cybersecurity_intrusion_train_preprocessed.csv`
- `cybersecurity_intrusion_test_preprocessed.csv`
- `preprocessor.joblib` (fitted on train split)

These files will be saved in `data/processed/`.

In [19]:
train_out = PROCESSED_DATA_DIR / "cybersecurity_intrusion_train_preprocessed.csv"
test_out = PROCESSED_DATA_DIR / "cybersecurity_intrusion_test_preprocessed.csv"
preproc_out = PROCESSED_DATA_DIR / "preprocessor.joblib"

train_processed.to_csv(train_out, index=False)
test_processed.to_csv(test_out, index=False)
joblib.dump(preprocessor, preproc_out)

print("Saved:")
print("-", train_out)
print("-", test_out)
print("-", preproc_out)


Saved:
- C:\Users\Asus\PycharmProjects\AIProject\data\processed\cybersecurity_intrusion_train_preprocessed.csv
- C:\Users\Asus\PycharmProjects\AIProject\data\processed\cybersecurity_intrusion_test_preprocessed.csv
- C:\Users\Asus\PycharmProjects\AIProject\data\processed\preprocessor.joblib


## 9) Preprocess the full dataset for final training

This step refits the preprocessing pipeline on the full dataset.
The resulting file is intended ONLY for final training or deployment,
and NOT for model evaluation.


In [20]:
preprocessor_full = clone(preprocessor)
preprocessor_full.fit(X)

X_full_p = preprocessor_full.transform(X)

full_feature_names = preprocessor_full.get_feature_names_out()
X_full_df = pd.DataFrame(X_full_p, columns=full_feature_names)

full_processed = pd.concat(
    [X_full_df, y.reset_index(drop=True)],
    axis=1
)

full_out = PROCESSED_DATA_DIR / "cybersecurity_intrusion_full_preprocessed.csv"
full_processed.to_csv(full_out, index=False)

print("Saved full dataset to:", full_out)


Saved full dataset to: C:\Users\Asus\PycharmProjects\AIProject\data\processed\cybersecurity_intrusion_full_preprocessed.csv


**Observation:**  
The preprocessing pipeline was refitted on the full dataset and applied consistently to generate a complete preprocessed version of the data. This file is intended for final model training or deployment purposes and should not be used for model evaluation.
